# 01 — Download & Benchmark Qwen3-8B (Q4_K_M)

**Qwen3-8B Q4_K_M** is the proven baseline: 5.03 GB on disk, **32.0 tok/s** on a free T4, only **5467 MiB VRAM**. This notebook downloads it, loads it onto the GPU, and benchmarks generation speed.

## 1. Install llama-cpp-python with CUDA

In [ ]:
%%capture
pip install llama-cpp-python huggingface_hub --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu124


## 2. Download the model from Hugging Face

In [ ]:
from huggingface_hub import hf_hub_download

repo_id = "Qwen/Qwen3-8B-GGUF"
filename = "qwen3-8b-q4_k_m.gguf"

model_path = hf_hub_download(repo_id=repo_id, filename=filename)
print(f"Model downloaded to: {model_path}")
import os
print(f"File size: {os.path.getsize(model_path) / 1024**3:.2f} GB")


## 3. Load the model onto the GPU

In [ ]:
from llama_cpp import Llama
import time

t0 = time.time()
llm = Llama(
    model_path=model_path,
    n_gpu_layers=-1,   # offload all layers to the T4
    n_ctx=4096,        # context window
    verbose=False,
)
print(f"Model loaded in {time.time()-t0:.1f}s")


## 4. Check VRAM usage

In [ ]:
!nvidia-smi --query-gpu=memory.used,memory.total --format=csv


## 5. Benchmark generation speed

In [ ]:
import time

prompt = "Explain the difference between supervised and unsupervised learning in three sentences."

# warmup
llm(prompt, max_tokens=16, verbose=False)

# timed run
n_tokens = 256
t0 = time.time()
resp = llm(prompt, max_tokens=n_tokens, verbose=False)
elapsed = time.time() - t0
tps = n_tokens / elapsed

print(f"Generated {n_tokens} tokens in {elapsed:.2f}s")
print(f"Throughput: {tps:.1f} tok/s")
print(f"\nSample output:\n{resp['choices'][0]['text'][:500]}")


## Expected results

| Metric | Value |
|--------|-------|
| File size | 5.03 GB |
| VRAM used | ~5467 MiB |
| Throughput | ~32.0 tok/s |

If you see numbers in this ballpark, your T4 setup is working correctly.